In [ ]:
import datetime as dt
import dask.dataframe as dd
from sqlalchemy import select, create_engine
from sqlalchemy.sql.expression import func
from sqlalchemy.sql.expression import literal_column, literal
from sqlalchemy.dialects.postgresql import INTERVAL
from dotenv import load_dotenv
import os
import mc_postgres_db.models as models
from sqlalchemy.orm import Session

load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

engine = create_engine(POSTGRES_URL)

In [ ]:
date: dt.date = dt.datetime.now(dt.timezone.utc).date() - dt.timedelta(days=1)
end = dt.datetime.combine(date, dt.time.min)
start = end - dt.timedelta(days=1)
start_naive = start.replace(tzinfo=None).replace(second=0, microsecond=0)
end_naive = end.replace(tzinfo=None).replace(second=0, microsecond=0)
print(f"Start: {start_naive}, End: {end_naive}")

In [ ]:
max_groups = 5
with Session(engine) as session:
    # Get all provider asset group id(s)
    provider_asset_group_ids = session.scalars(
        select(models.ProviderAssetGroup.id).limit(max_groups)
    ).all()
print(
    f"Provider asset group ids (count: {len(provider_asset_group_ids)}): {provider_asset_group_ids}"
)

In [ ]:
# Create subquery that generates minutely timestamps using PostgreSQL's generate_series
start_str = start_naive.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")
end_str = end_naive.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")
time_frame_subquery = select(
    func.generate_series(
        literal_column(start_str),
        literal_column(end_str),
        func.cast(literal("1 minute"), INTERVAL),
    ).label("timestamp")
).subquery("time_frame")

# Create the sub-query for all provider asset group members.
provider_asset_group_members_subquery = (
    select(
        models.ProviderAssetGroupMember.provider_asset_group_id,
        models.ProviderAssetGroupMember.order,
        models.ProviderAssetGroupMember.provider_id,
        models.ProviderAssetGroupMember.from_asset_id,
        models.ProviderAssetGroupMember.to_asset_id,
    )
    .where(
        models.ProviderAssetGroupMember.provider_asset_group_id.in_(
            provider_asset_group_ids
        )
    )
    .subquery("provider_asset_group_members")
)

# Combine the time-frame with the provider asset group members into a dask dataframe.
full_frame = dd.read_sql_query(
    select(
        time_frame_subquery.c.timestamp,
        provider_asset_group_members_subquery.c.provider_asset_group_id,
        provider_asset_group_members_subquery.c.provider_id,
        provider_asset_group_members_subquery.c.from_asset_id,
        provider_asset_group_members_subquery.c.to_asset_id,
    )
    .select_from(time_frame_subquery, provider_asset_group_members_subquery)
    .order_by(time_frame_subquery.c.timestamp),
    engine.url.render_as_string(hide_password=False),
    index_col="timestamp",
    bytes_per_chunk="512 MiB",
)

# Get the market data Dask dataframe.
market_data = dd.read_sql_query(
    select(
        models.ProviderAssetMarket.timestamp,
        models.ProviderAssetMarket.provider_id,
        models.ProviderAssetMarket.from_asset_id,
        models.ProviderAssetMarket.to_asset_id,
        models.ProviderAssetMarket.close,
    ).where(models.ProviderAssetMarket.timestamp.between(start_naive, end_naive)),
    engine.url.render_as_string(hide_password=False),
    index_col="timestamp",
    bytes_per_chunk="512 MiB",
)

# As-of join the market data with the full frame.
full_frame = dd.merge_asof(
    full_frame,
    market_data,
    right_index=True,
    left_index=True,
    by=["provider_id", "from_asset_id", "to_asset_id"],
)

# Compute the full frame.
full_frame.compute()